In [ ]:
library("Seurat")
library("tidyverse")
library('harmony')
library('ggpubr')
library('msigdbr')

In [ ]:
# ltc_palettes package
main_celltype_colors <- c('T_Cell' = '#9b2226', 'NK' = '#D55D4C', 'B_Cell' = '#ca6702', 'Plasma' = '#ee9b00',
                          'Macro' = '#005F73', 'Mono' = '#0a9396', 'DC' = '#94d2bd', 'Mast' = '#66679C', 'Prolif_Cell' = '#e9d8a6',
                          'ILC' = "#B84848")

In [ ]:
imm_subcelltype_colors <- c(Memory_B = "#2d6037", Naive_B = "#64AE59", Plasma_B = "#839098", CCR7_CD4_Tnaive = "#ECA8A9",
                            GZMK_CD8_Tem = "#74AED4", ZNF683_CD8_Trm = "#67ADB7",CXCR6_CD4_Trm = "#E4A6BD", ISG15_CD8_Teffector = '#B3B2B3',
                            MAIT = "#F3D8E1", FOXP3_CD4_Treg = "#009170", CXCL13_CD4_Tex = "#78A040", gdT = '#839098',
                            GZMB_CD8_Teffector = "#2E75AB", CXCL13_CD8_Tex = "#009393",FGFBP2_NK = "#B06E3C", XCL1_NK = "#5AA2DA",
                            NKT = "#FBD8A2", ILC = "#B84848", Mast = "#6567A0", Undetermined = "#87C3EC",
                            pDC = "#BDE6FA", cDC1 = "#D7EFFB", cDC2 = "#6EB1DE", LAMP3_DC = "#92C2DD", Neutrophil = "#4A94C6",
                            PPARG_Mono = "#FADED2", CD16_Mono = "#FAC7B3", PPARG_Macro = "#F0A29B", SPP1_Macro = "#B389B9",
                            LGMN_Macro = "#CC7892", FABP4_Macro = "#E2A2B3", CD14_Mono = "#F3C6C1", MIF_Macro = "#89558D")

In [ ]:
transparent_bg <- theme(panel.background = element_rect(fill = NA, colour = NA),
                        plot.background = element_rect(fill = NA, colour = NA),
                        legend.box.background = element_rect(fill = NA, colour = NA),
                        legend.background = element_rect(fill = NA, colour = NA))

In [ ]:
tumor_dataset_collection <- "/public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection"
obj <- "/public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects"
figures <- "/public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/figures"


# Data load and preprocess

In [ ]:
##load the data from raw fastq based starsolo of crm_2024_Hanjie_Li dataset (MARS-seq)
Crm2024_dir <- paste0(tumor_dataset_collection, "/crm_2024_Hanjie_Li/final_expression")
Crm2024_SeuratObj <- CreateSeuratObject(Read10X(Crm2024_dir), project = "crm_2024_Hanjie", min.cells = 10)

In [ ]:
Crm2024_SeuratObj

In [ ]:
## load the metadata
crm2024_CellMetadata <- read.table(paste0(Crm2024_dir, '/cell_metadata.tsv.gz'), header = TRUE)

crm2024_PlateMetadata <- read.csv(paste0(Crm2024_dir, '/HRA002961_metadata_with_age.csv'), header = TRUE) %>% mutate(Plate = `Run.accession`)

crm2024_Metadata <- crm2024_CellMetadata %>% 
    rename(Plate = Sample) %>% 
    left_join(crm2024_PlateMetadata, by = "Plate") %>%
    column_to_rownames('Cell') %>%
    mutate(Age_group = case_when(Age <= 40 ~ "Young", Age >= 55 ~ "Old", .default = "Middle") )

Crm2024_SeuratObj <- AddMetaData(Crm2024_SeuratObj, metadata = crm2024_Metadata)

In [ ]:
Crm2024_SeuratObj[[]] %>% colnames()

In [ ]:
Crm2024_SeuratObj$`Age_group` %>% unique()

In [ ]:
## filter: age_group = young or old; tissue_normalized = tumor; Smoking = no (all young samples in the dataset is non-smoker)
Crm2024_SeuratObj <- subset(Crm2024_SeuratObj, subset = (tissue_normalized == "tumor" & Age_group %in% c("Young", "Old") & Smoking == "no"))

In [ ]:
Crm2024_SeuratObj

In [ ]:
summary(Crm2024_SeuratObj$nFeature_RNA);
summary(Crm2024_SeuratObj$nCount_RNA);

In [ ]:
## gene filter
Crm2024_SeuratObj <- Crm2024_SeuratObj[!grepl(pattern = "^MT-", x = rownames(Crm2024_SeuratObj)),]
Crm2024_SeuratObj <- Crm2024_SeuratObj[!grepl(pattern = "^RP([0-9]+-|[LS])", x = rownames(Crm2024_SeuratObj)),]


In [ ]:
Crm2024_SeuratObj

In [ ]:
table(Crm2024_SeuratObj$T_stage, Crm2024_SeuratObj$Age_group)

# Integration

In [ ]:
#normalization
Crm2024_SeuratObj <- Crm2024_SeuratObj %>% NormalizeData(verbose = FALSE)

In [ ]:
# find variable genes in each sample
Crm2024_SeuratObjList <- SplitObject(object = Crm2024_SeuratObj, split.by = 'Sample')

Crm2024_SeuratObjList <- lapply(Crm2024_SeuratObjList, function(obj) {
    obj <- NormalizeData(obj)
    obj <- FindVariableFeatures(obj, selection.method = "vst", nfeatures = 2000)
    return(obj)
})

# 
fvf_features <- SelectIntegrationFeatures(object.list = Crm2024_SeuratObjList, nfeatures = 2000)

# 
VariableFeatures(Crm2024_SeuratObj) <- fvf_features

In [ ]:
#run scale_data
Crm2024_SeuratObj <- ScaleData(Crm2024_SeuratObj, verbose = T)

In [ ]:
#RunPCA
Crm2024_SeuratObj <- RunPCA(Crm2024_SeuratObj, features = as.character(VariableFeatures(Crm2024_SeuratObj)), npcs = 50, verbose = FALSE)

In [ ]:
# run harmony
options(repr.plot.height = 6, repr.plot.width = 6)
Crm2024_SeuratObj <- RunHarmony(object = Crm2024_SeuratObj, group.by.vars = 'Sample', max.iter.harmony = 20, plot_convergence = T)

In [ ]:
# UMAP
Crm2024_SeuratObj <- RunUMAP(Crm2024_SeuratObj, dims = 1:50, reduction = "harmony", verbose = F )

In [ ]:
options(repr.plot.height = 6, repr.plot.width = 12)
DimPlot(object = Crm2024_SeuratObj, reduction = 'umap', group.by = 'Sample')

In [ ]:
options(repr.plot.height = 6, repr.plot.width = 8)
DimPlot(object = Crm2024_SeuratObj, reduction = 'umap', group.by = 'Age_group')

In [ ]:
options(repr.plot.height = 6, repr.plot.width = 12)
DimPlot(object = Crm2024_SeuratObj, reduction = 'umap', group.by = 'Age_group', split.by = 'Age_group')

In [ ]:
Crm2024_SeuratObj$patient_ID %>% unique %>% length()

# Clustering and annotation

In [ ]:
# FindNeighbors
Crm2024_SeuratObj <- FindNeighbors(Crm2024_SeuratObj, dims = 1:50, reduction = "harmony", verbose = F)

In [ ]:
# FindClusters
options(repr.plot.height = 8, repr.plot.width = 8)
for (i in c(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1)) {
  Crm2024_SeuratObj <- FindClusters(Crm2024_SeuratObj, resolution = i, verbose = F)
  print(DimPlot(Crm2024_SeuratObj, reduction = "umap", label = T) + labs(title = paste0("resolution: ", i)))
}

In [ ]:
options(repr.plot.height =8, repr.plot.width = 8)

FeaturePlot(object = Crm2024_SeuratObj, features = c("CXCL13"), order = T)

In [ ]:
options(repr.plot.height =8, repr.plot.width = 8)

FeaturePlot(object = Crm2024_SeuratObj, features = c("CD4"), order = T)

In [ ]:
options(repr.plot.height =24, repr.plot.width = 24)

imm_markers <- c('TRAC', 'CD3D',
                 'KLRF1', 'NCR1',
                 'CD79A', 'MS4A1', 'JCHAIN',
                 'CD1C', 'CLEC9A', 'FSCN1', 'LILRA4',
                 'C1QC', 'CD68',
                 'FCN1', 'CD14', 'FCGR3A', 'S100A8',
                 'CSF3R', 'FCGR3B',
                 'KIT')

FeaturePlot(object = Crm2024_SeuratObj, features = imm_markers, order = T)

In [ ]:
options(repr.plot.height =8, repr.plot.width = 8)
DimPlot(Crm2024_SeuratObj, group.by = 'RNA_snn_res.1', label = T)

In [ ]:
#find ImmMarkers_Crm2024_res1_presto with wilcox test in presto
ImmMarkers_Crm2024_res1_presto <- presto::wilcoxauc(X = Crm2024_SeuratObj, group_by = 'RNA_snn_res.1')

In [ ]:
ImmMarkers_Crm2024_res1_presto %>% filter(group == '10') %>% arrange(-auc) %>% pull(feature) %>% head(100)

In [ ]:
print('done')

In [ ]:
#immcluster annotation_res1
options(repr.plot.height = 10, repr.plot.width = 12)
Idents(Crm2024_SeuratObj) <- Crm2024_SeuratObj$`RNA_snn_res.1`
## rename clusters
Immcluster_anno <- c("0" = "T_Cell", "1" = "NK", "2" = "T_Cell", "3" = "T_Cell", "4" = "B_Cell",
                     "5" = "T_Cell", "6" = "T_Cell", "7" = "T_Cell", "8" = "T_Cell",
                     "9" = "DC", "10" = "ILC", "11" = "T_Cell", "12" = "T_Cell", "13" = "B_Cell", "14" = "Macro", "15" = "T_Cell",
                     "16" = "Prolif_Cell", "17" = "Mono", "18" = "Mono", "19" = "Plasma", "20" = "Mast", "21" = "T_Cell", "22" = "DC", "23" = "DC")

Crm2024_SeuratObj <- RenameIdents(Crm2024_SeuratObj, Immcluster_anno)
Crm2024_SeuratObj$Immcluster_res1 <- Idents(Crm2024_SeuratObj)

DimPlot(Crm2024_SeuratObj, label = T, pt.size = 0.5, label.size = 8, repel = T, reduction = 'umap', group.by = 'Immcluster_res1') +
    theme(plot.title = element_text(size = 30),
          legend.text = element_text(size = 20),
          legend.key.size = unit(0.5, "inches")) +
    guides(colour = guide_legend(override.aes = list(size = 5)))

In [ ]:
DimPlot(Crm2024_SeuratObj, group.by = 'Immcluster_res1', label = F, pt.size = 0.5, raster=FALSE, shuffle=T) +
    scale_color_manual(values = main_celltype_colors) +
    labs(title = NULL) +
    theme_classic(base_size = 12) +
    transparent_bg +
    theme(axis.text = element_blank(),
          axis.line = element_blank(),
          axis.ticks = element_blank(),
          axis.title = element_blank(),
          legend.key.height = unit(x = 0.5, units = 'in'),
          legend.key.size = unit(x = 0.5, units = 'in'))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Crm2024_Imm_cell_UMAP.pdf'), device = 'pdf', width = 10, height = 10, bg = 'transparent')
ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Crm2024_Imm_cell_UMAP.png'), device = 'png', width = 10, height = 10, dpi = 300, bg = 'transparent')

In [ ]:
saveRDS(Crm2024_SeuratObj, paste0(obj, '/Crm2024_SeuratObj.rds'))

In [ ]:
Crm2024_SeuratObj <- readRDS(paste0(obj, '/Crm2024_SeuratObj.rds'))

In [ ]:
Crm2024_SeuratObj$patient_ID %>% unique()

In [ ]:
Crm2024_SeuratObj$Sample %>% unique()

In [ ]:
Crm2024_SeuratObj

# T_Cell_analysis

### Subset

In [ ]:
Idents(Crm2024_SeuratObj) <- Crm2024_SeuratObj$`Immcluster_res1`
levels(Crm2024_SeuratObj)

In [ ]:
# subset T_Cell
Idents(Crm2024_SeuratObj) <- Crm2024_SeuratObj$`Immcluster_res1`
Crm2024_TCell_SeuratObj <- subset(Crm2024_SeuratObj, idents = c('T_Cell'))
Crm2024_TCell_SeuratObj

### Clustring and annotation analysis

In [ ]:
# UMAP
Crm2024_TCell_SeuratObj <- RunUMAP(Crm2024_TCell_SeuratObj, dims = 1:50, reduction = "harmony", verbose = F )


In [ ]:
# FindNeighbors
Crm2024_TCell_SeuratObj <- FindNeighbors(Crm2024_TCell_SeuratObj, dims = 1:50, reduction = "harmony", verbose = F)

In [ ]:
# FindClusters
options(repr.plot.width = 8, repr.plot.height = 8)
for (i in c(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1)) {
  Crm2024_TCell_SeuratObj <- FindClusters(Crm2024_TCell_SeuratObj, resolution = i, verbose = F)
  print(DimPlot(Crm2024_TCell_SeuratObj, reduction = "umap", label = T) + labs(title = paste0("resolution: ", i)))
}

In [ ]:
options(repr.plot.height =24, repr.plot.width = 24)
T_markers1 <- c('TRAC', 'CD3E', 'CD4', 'CD8A', 'TCF7', 'SELL', 'LEF1', 'CCR7', 'IL7R',
                'CD27', 'CD28', 'MAL', 'KLF2', 'PIK3IP1', 'TRDC', 'TRGC2')

FeaturePlot(object = Crm2024_TCell_SeuratObj, features = T_markers1, order = T)

In [ ]:
options(repr.plot.height =48, repr.plot.width = 24)
T_markers2 <- c(
    'CXCR6', 'CD69', 'IL7R', 'KLRB1', 'PTGER4',
    'IFNG', 'GZMA', 'GZMB', 'GZMK', 'GNLY', 'PRF1', 'NKG7',
    'ZNF683', 'ITGAE', 'RBPJ',
    'LAG3', 'TIGIT', 'PDCD1', 'HAVCR2', 'CTLA4', 'CXCL13','TOX',
    'IL2RA', 'FOXP3', 'IKZF2',
    'IFITM3', 'IFI27', 'ISG15', 'ISG20',
    'SLC4A10', 'IL12RB', 'IL18R1', 'PLZF')

FeaturePlot(object = Crm2024_TCell_SeuratObj, features = T_markers2, order = T)

In [ ]:
# get the multiple_subcluster for c7_res0.4
Idents(Crm2024_TCell_SeuratObj) <- Crm2024_TCell_SeuratObj$`RNA_snn_res.0.4`
Crm2024_TCell_SeuratObj <- FindSubCluster(object = Crm2024_TCell_SeuratObj, cluster = 7, subcluster.name = 'RNA_snn_res.0.4_subclus', graph.name = 'RNA_snn', resolution = 0.7)

In [ ]:
# get the multiple_subcluster for c7c0_res0.4
Idents(Crm2024_TCell_SeuratObj) <- Crm2024_TCell_SeuratObj$`RNA_snn_res.0.4_subclus`
Crm2024_TCell_SeuratObj <- FindSubCluster(object = Crm2024_TCell_SeuratObj, cluster = 0, subcluster.name = 'RNA_snn_res.0.4_subclus', graph.name = 'RNA_snn', resolution = 0.2)

In [ ]:
options(repr.plot.height =8, repr.plot.width = 8)
DimPlot(Crm2024_TCell_SeuratObj, group.by = 'RNA_snn_res.0.4_subclus', label = T)

In [ ]:
#find TMarkers_Crm2024_res0.4_presto with wilcox test in presto
TMarkers_Crm2024_res0.4_presto <- presto::wilcoxauc(X = Crm2024_TCell_SeuratObj, group_by = 'RNA_snn_res.0.4_subclus')

In [ ]:
TMarkers_Crm2024_res0.4_presto %>% filter(group == '7_1') %>% arrange(-auc) %>% head(60)

In [ ]:
#Tcluster annotation_res0.4
options(repr.plot.height = 10, repr.plot.width = 12)
Idents(Crm2024_TCell_SeuratObj) <- Crm2024_TCell_SeuratObj$`RNA_snn_res.0.4_subclus`
## rename clusters
Tcluster_anno <- c("0_0" = "GZMB_CD8_Teffector", "0_1" = "NKT", "1" = "CXCR6_CD4_Trm", "2" = "GZMK_CD8_Tem", "3" = "ZNF683_CD8_Trm", "4" = "CCR7_CD4_Tnaive",
                    "5" = "MAIT", "6" = "FOXP3_CD4_Treg", "7_0" = "CXCL13_CD8_Tex", "7_1" = "CXCL13_CD4_Tex", "7_2" = "CXCL13_CD8_Tex", "8" = "ISG15_CD8_Teffector", "9" = "gdT")


Crm2024_TCell_SeuratObj <- RenameIdents(Crm2024_TCell_SeuratObj, Tcluster_anno)
Crm2024_TCell_SeuratObj$Tcluster_res0.4 <- Idents(Crm2024_TCell_SeuratObj)

DimPlot(Crm2024_TCell_SeuratObj, label = T, pt.size = 0.5, label.size = 6, repel = T, reduction = 'umap', group.by = 'Tcluster_res0.4') +
    theme(plot.title = element_text(size = 30),
          legend.text = element_text(size = 20),
          legend.key.size = unit(0.5, "inches")) +
    guides(colour = guide_legend(override.aes = list(size = 5)))

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 8)
T_subcluster <- c('CCR7_CD4_Tnaive', 'CXCR6_CD4_Trm', 'CXCL13_CD4_Tex', 'FOXP3_CD4_Treg',
                 'GZMK_CD8_Tem', 'GZMB_CD8_Teffector', 'ZNF683_CD8_Trm', 'CXCL13_CD8_Tex',
                 'ISG15_CD8_Teffector', 'MAIT', 'gdT', 'NKT')

DimPlot(Crm2024_TCell_SeuratObj, group.by = 'Tcluster_res0.4', label = F, pt.size = 0.5, raster=FALSE, shuffle=F) +
    scale_color_manual(values = imm_subcelltype_colors, limits = T_subcluster) +
    labs(title = NULL) +
    theme_classic(base_size = 12) +
    transparent_bg +
    theme(axis.text = element_blank(),
          axis.line = element_blank(),
          axis.ticks = element_blank(),
          axis.title = element_blank(),
#          legend.position = 'none',
          legend.key.height = unit(x = 0.5, units = 'in'),
          legend.key.size = unit(x = 0.5, units = 'in'))

#ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Crm2024_TCell_UMAP.pdf'), device = 'pdf', width = 6, height = 6, bg = 'transparent')
ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Crm2024_TCell_UMAP.png'), device = 'png', width = 10, height = 8, dpi = 300, bg = 'transparent')

In [ ]:
T_markers <- c('TRAC', 'CD3E', 'CD4', 'CD8A',
    'TCF7', 'CCR7',
    'CXCR6', 'IL7R', 'KLRB1',
    'IFNG', 'GZMA', 'GZMB', 'GZMK', 'PRF1', 'NKG7',
    'ZNF683', 'ITGAE', 'RBPJ',
    'PDCD1', 'CTLA4', 'CXCL13','TOX',
    'FOXP3', 'IKZF2',
    'ISG15', 'SLC4A10')


In [ ]:
#T_Cells_markers
options(repr.plot.width = 14, repr.plot.height = 10)
DotPlot(Crm2024_TCell_SeuratObj, features = T_markers, group.by = 'Tcluster_res0.4', scale = T, col.max = 3) + 
    scale_y_discrete(limits=c('CCR7_CD4_Tnaive', 'CXCR6_CD4_Trm', 'CXCL13_CD4_Tex', 'FOXP3_CD4_Treg',
                              'GZMK_CD8_Tem', 'GZMB_CD8_Teffector', 'ZNF683_CD8_Trm', 'CXCL13_CD8_Tex',
                              'ISG15_CD8_Teffector', 'MAIT', 'gdT', 'NKT')) +
    scale_color_gradient(low = '#E2E1EF', high = '#C23339', guide = guide_colorbar(order = 1), limits = c(-2,2), oob = scales::squish) +
    scale_size_area(max_size = 10, guide = guide_legend(order = 2)) +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(panel.border = element_rect(linewidth = 1, fill = NA, color = 'black'),
          legend.title = element_text(size = 20),
          legend.position = 'top',
          axis.text = element_text(colour = 'black'),
          axis.line = element_blank(),
          axis.text.x = element_text(angle = 90),
          axis.title.x = element_blank(),
          axis.title.y = element_blank())

#C23339
ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Crm2024_TCells_markers_Dotplot.pdf'), device = 'pdf', width = 12, height = 8, bg = 'transparent')

In [ ]:
saveRDS(Crm2024_TCell_SeuratObj, paste0(obj, '/Crm2024_TCell_SeuratObj.rds'))

In [ ]:
Crm2024_TCell_SeuratObj <- readRDS(paste0(obj, '/Crm2024_TCell_SeuratObj.rds'))

## Signature score analysis

In [ ]:
# curated gene set
## cd8_tumor_reactivity_signature ('A phenotypic signature that identifies neoantigen-reactive T cells in fresh human lung cancers')
CD8_T_tumor_reactivity_signature <- c('CXCL13', 'ENTPD1', 'BATF', 'GZMB', 'CD27', 'TIGIT', 'PHLDA1',
                                    'CD74', 'HLA-DMA', 'HLA-DRA', 'HLA-DRB1', 'HLA-DPB1', 'CD3D', 'CD82',
                                    'ARL3', 'HMOX1', 'ALOX5AP', 'DUSP4', 'CARS', 'LSP1', 'CCND2',
                                    'TPI1', 'GAPDH', 'ITM2A', 'HMGN3', 'CHST12', 'NAP1L4')

## cd4_tumor_reactivity_signature ('A phenotypic signature that identifies neoantigen-reactive T cells in fresh human lung cancers')
CD4_T_tumor_reactivity_signature <- c('CXCL13', 'NR3C1', 'ADGRG1', 'NMG', 'ITM2A', 'ETV7', 'COTL1', 'B2M', 'IGFL2')

In [ ]:
unique(Crm2024_TCell_SeuratObj$`Tcluster_res0.4`)

In [ ]:
# subset CD4T and CD8T
Idents(Crm2024_TCell_SeuratObj) <- Crm2024_TCell_SeuratObj$`Tcluster_res0.4`
Crm2024_CD8TCell_SeuratObj <- subset(Crm2024_TCell_SeuratObj,
                                       idents = c('ZNF683_CD8_Trm', 'GZMK_CD8_Tem', 'GZMB_CD8_Teffector', 'CXCL13_CD8_Tex', 'ISG15_CD8_Teffector'))
Crm2024_CD4TCell_SeuratObj <- subset(Crm2024_TCell_SeuratObj,
                                       idents = c('CCR7_CD4_Tnaive', 'CXCR6_CD4_Trm', 'FOXP3_CD4_Treg', 'CXCL13_CD4_Tex'))

In [ ]:
# module score
Crm2024_CD8TCell_SeuratObj <- AddModuleScore(object = Crm2024_CD8TCell_SeuratObj, features = list(CD8_T_tumor_reactivity_signature),
                                               seed = 42, name = 'CD8_T_Tumor_Reactivity_Score')

Crm2024_CD4TCell_SeuratObj <- AddModuleScore(object = Crm2024_CD4TCell_SeuratObj, features = list(CD4_T_tumor_reactivity_signature),
                                               seed = 42, name = 'CD4_T_Tumor_Reactivity_Score')

In [ ]:
#CD8_T_Tumor_Reactivity_Score('A phenotypic signature that identifies neoantigen-reactive T cells in fresh human lung cancers')
options(repr.plot.width = 12, repr.plot.height = 10)
FetchData(Crm2024_CD8TCell_SeuratObj, vars = c('Tcluster_res0.4', 'Sample', 'CD8_T_Tumor_Reactivity_Score1')) %>%
    ggplot(mapping = aes(x = Tcluster_res0.4, y = CD8_T_Tumor_Reactivity_Score1, fill = Tcluster_res0.4)) +
    geom_violin(linewidth=1, position = position_dodge(0.7), alpha = 0.9) +
    geom_boxplot(notch=T, width=0.2, linewidth=1, position = position_dodge(0.7)) +
    stat_compare_means(mapping = aes(x = Tcluster_res0.4, y = CD8_T_Tumor_Reactivity_Score1), label = 'p.format', method = 'wilcox.test',
                       comparisons = list(c('CXCL13_CD8_Tex', 'ISG15_CD8_Teffector'), c('GZMB_CD8_Teffector', 'CXCL13_CD8_Tex'),
                                          c('GZMK_CD8_Tem', 'CXCL13_CD8_Tex'), c('ZNF683_CD8_Trm', 'CXCL13_CD8_Tex')),
                       label.y = c(1.6, 1.8, 2, 2.2), tip.length = 0.015) +
    scale_fill_manual(values = imm_subcelltype_colors, guide = guide_legend(title = element_blank(), nrow = 2)) +
    scale_x_discrete(limits = c('ISG15_CD8_Teffector', 'GZMB_CD8_Teffector', 'GZMK_CD8_Tem', 'ZNF683_CD8_Trm', 'CXCL13_CD8_Tex')) +
    geom_hline(yintercept = 0, linetype='dashed') +
    xlab(label = NULL) +
    ylab(label = 'CD8+ T Cell Tumor Reactivity Score') +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(legend.position = 'none',
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
          axis.text.x = element_text(angle = 0),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Crm2024_CD8TCell_Tumor_Reactivity_Score_VlnPlot.pdf'), device = 'pdf', bg = 'transparent', width = 10, height = 10)

In [ ]:
#CD4_T_Tumor_Reactivity_Score('A phenotypic signature that identifies neoantigen-reactive T cells in fresh human lung cancers')
options(repr.plot.width = 12, repr.plot.height = 10)
FetchData(Crm2024_CD4TCell_SeuratObj, vars = c('Tcluster_res0.4', 'Sample', 'CD4_T_Tumor_Reactivity_Score1')) %>%
    ggplot(mapping = aes(x = Tcluster_res0.4, y = CD4_T_Tumor_Reactivity_Score1, fill = Tcluster_res0.4)) +
    geom_violin(linewidth=1, position = position_dodge(0.7), alpha = 0.9) +
    geom_boxplot(notch=T, width=0.2, linewidth=1, position = position_dodge(0.7)) +
    stat_compare_means(mapping = aes(x = Tcluster_res0.4, y = CD8_T_Tumor_Reactivity_Score1), label = 'p.format', method = 'wilcox.test',
                       comparisons = list(c('CCR7_CD4_Tnaive', 'CXCL13_CD4_Tex'), c('CXCR6_CD4_Trm', 'CXCL13_CD4_Tex'),
                                          c('FOXP3_CD4_Treg', 'CXCL13_CD4_Tex')),
                       label.y = c(2, 2.2, 2.4, 2.6), tip.length = 0.015) +
    scale_fill_manual(values = imm_subcelltype_colors, guide = guide_legend(title = element_blank(), nrow = 2)) +
    scale_x_discrete(limits = c('CCR7_CD4_Tnaive', 'CXCR6_CD4_Trm', 'CXCL13_CD4_Tex', 'FOXP3_CD4_Treg')) +
    geom_hline(yintercept = 0, linetype='dashed') +
    xlab(label = NULL) +
    ylab(label = 'CD4+ T Cell Tumor Reactivity Score') +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(legend.position = 'none',
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
          axis.text.x = element_text(angle = 0),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Crm2024_CD4TCell_Tumor_Reactivity_Score_VlnPlot.pdf'), device = 'pdf', bg = 'transparent', width = 10, height = 10)

In [ ]:
Crm2024_CD8TCell_SeuratObj[[]] %>% colnames()

In [ ]:
Crm2024_CD8TCell_SeuratObj$`Age_group` %>% unique()

In [ ]:
#CXCL13_CD8_Tumor_Reactivity_Score(only diseased)('A phenotypic signature that identifies neoantigen-reactive T cells in fresh human lung cancers')
FetchData(Crm2024_CD8TCell_SeuratObj, vars = c('Tcluster_res0.4', 'Age_group', 'CD8_T_Tumor_Reactivity_Score1')) %>%
    filter(Tcluster_res0.4 == 'CXCL13_CD8_Tex') %>%
    mutate(Age_group = factor(Age_group, levels = c('Young', 'Old'))) %>%
    ggplot(mapping = aes(x = Age_group, y = CD8_T_Tumor_Reactivity_Score1, fill = Age_group)) +
    geom_violin(linewidth=1, position = position_dodge(0.8), alpha = 0.9) +
    geom_boxplot(notch=T, width=0.2, linewidth=1, position = position_dodge(0.8)) +
    stat_compare_means(comparisons = list(c('Young', 'Old')), label = 'p.format', method = 'wilcox.test') +
    geom_hline(yintercept = 0, linetype='dashed') +
    scale_fill_manual(values = age_group_color, guide = guide_legend(title = 'Age Group', nrow = 1)) +
    xlab(label = NULL) +
    ylab(label = 'CD8+ T Cell Tumor Reactivity Score') +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(legend.position = 'none',
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
          axis.text.x = element_text(angle = 0),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Crm2024_CD8TCell_CXCL13_CD8_T_Cells_Tumor_Reactivity_Score_VlnPlot.pdf'), device = 'pdf', bg = 'transparent', width = 10, height = 10)

In [ ]:
#CD4_T_Tumor_Reactivity_Score('A phenotypic signature that identifies neoantigen-reactive T cells in fresh human lung cancers')
FetchData(Crm2024_CD4TCell_SeuratObj, vars = c('Tcluster_res0.4', 'Age_group', 'CD4_T_Tumor_Reactivity_Score1')) %>%
    filter(Tcluster_res0.4 == 'CXCL13_CD4_Tex') %>%
    mutate(Age_group = factor(Age_group, levels = c('Young', 'Old'))) %>%
    ggplot(mapping = aes(x = Age_group, y = CD4_T_Tumor_Reactivity_Score1, fill = Age_group)) +
    geom_violin(linewidth=1, position = position_dodge(0.8), alpha = 0.9) +
    geom_boxplot(notch=T, width=0.2, linewidth=1, position = position_dodge(0.8)) +
    stat_compare_means(comparisons = list(c('Young', 'Old')), label = 'p.format', method = 'wilcox.test') +
    geom_hline(yintercept = 0, linetype='dashed') +
    scale_fill_manual(values = age_group_color, guide = guide_legend(title = 'Age Group', nrow = 1)) +
    xlab(label = NULL) +
    ylab(label = 'CD4+ T Cell Tumor Reactivity Score') +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(legend.position = 'none',
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
          axis.text.x = element_text(angle = 0),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Crm2024_CD4TCell_CXCL13_CD4_T_Cells_Tumor_Reactivity_Score_VlnPlot.pdf'), device = 'pdf', bg = 'transparent', width = 10, height = 10)

# B_Cell_analysis

### Subset

In [ ]:
Idents(Crm2024_SeuratObj) <- Crm2024_SeuratObj$`Immcluster_res1`
levels(Crm2024_SeuratObj)

In [ ]:
# subset B_Cell
Idents(Crm2024_SeuratObj) <- Crm2024_SeuratObj$`Immcluster_res1`
Crm2024_BCell_SeuratObj <- subset(Crm2024_SeuratObj, idents = c('B_Cell'))
Crm2024_BCell_SeuratObj

### Clustring and annotation analysis

In [ ]:
# UMAP
Crm2024_BCell_SeuratObj <- RunUMAP(Crm2024_BCell_SeuratObj, dims = 1:50, reduction = "harmony", verbose = F )


In [ ]:
# FindNeighbors
Crm2024_BCell_SeuratObj <- FindNeighbors(Crm2024_BCell_SeuratObj, dims = 1:50, reduction = "harmony", verbose = F)

In [ ]:
# FindClusters
options(repr.plot.width = 8, repr.plot.height = 8)
for (i in c(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1)) {
  Crm2024_BCell_SeuratObj <- FindClusters(Crm2024_BCell_SeuratObj, resolution = i, verbose = F)
  print(DimPlot(Crm2024_BCell_SeuratObj, reduction = "umap", label = T) + labs(title = paste0("resolution: ", i)))
}

In [ ]:
options(repr.plot.height =12, repr.plot.width = 12)
B_markers <- c('TCL1A', 'CD27', 'CD83', 'BCL6')

FeaturePlot(object = Crm2024_BCell_SeuratObj, features = B_markers, order = T)

In [ ]:
#Bcluster annotation_res0.2
options(repr.plot.height = 6, repr.plot.width = 8)
Idents(Crm2024_BCell_SeuratObj) <- Crm2024_BCell_SeuratObj$`RNA_snn_res.0.2`
## rename clusters
Bcluster_anno <- c("0" = "Memory_B", "1" = "Naive_B", "2" = "GC_B")

Crm2024_BCell_SeuratObj <- RenameIdents(Crm2024_BCell_SeuratObj, Bcluster_anno)
Crm2024_BCell_SeuratObj$Bcluster_res0.2 <- Idents(Crm2024_BCell_SeuratObj)

DimPlot(Crm2024_BCell_SeuratObj, label = T, pt.size = 0.5, label.size = 6, repel = T, reduction = 'umap', group.by = 'Bcluster_res0.2') +
    theme(plot.title = element_text(size = 30),
          legend.text = element_text(size = 20),
          legend.key.size = unit(0.5, "inches")) +
    guides(colour = guide_legend(override.aes = list(size = 5)))

In [ ]:
B_subcluster <- c('Naive_B', 'Memory_B')

DimPlot(Crm2024_BCell_SeuratObj, group.by = 'Bcluster_res0.2', label = F, pt.size = 1.5, raster=FALSE, shuffle=F) +
    scale_color_manual(values = imm_subcelltype_colors, limits = B_subcluster) +
    labs(title = NULL) +
    theme_classic(base_size = 12) +
    transparent_bg +
    theme(axis.text = element_blank(),
          axis.line = element_blank(),
          axis.ticks = element_blank(),
          axis.title = element_blank(),
          legend.position = 'none',
          legend.key.height = unit(x = 0.5, units = 'in'),
          legend.key.size = unit(x = 0.5, units = 'in'))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Crm2024_BCell_UMAP.png'), device = 'png', width = 6, height = 6, dpi = 300, bg = 'transparent')

In [ ]:
saveRDS(Crm2024_BCell_SeuratObj, paste0(obj, '/Crm2024_BCell_SeuratObj.rds'))

## DEGs analysis

In [ ]:
# MHCII_genes
MHCII_genes <- c('HLA-DRA', 'HLA-DRB5', 'HLA-DRB1', 'HLA-DQA1',
                 'HLA-DQB1', 'HLA-DQA2', 'HLA-DMB', 'HLA-DMA',
                 'HLA-DPA1', 'HLA-DPB1', 'HLA-DPB2', 'HLA-DRB6')

In [ ]:
# B_function_genes
#B_function_genes <- c(MHCII_genes, 'CTSD', 'TAP1', 'B2M', 'PSME1', 'PSME2',
#                      'IGHD','IGHM', 'IGHA1', 'IGHA2', 'IGHG1', 'IGHG2', #IgG
#                      'ISG20', 'IFITM1', 'IFI16', 'IFITM2','IRF8', #IFN
#                      'HSPA1A', 'HSPA1B', 'TUBB4B', 'GAPDH', 'MYC', # proliferating
#                      'KMT2C', 'CREBBP', 'KDM5A', 'JARID2', 'TET2')

# B_function_genes
#B_function_genes <- c(MHCII_genes, 'CTSD', 'TAP1', 'B2M', 'PSME1', 'PSME2',
#                      'IGHD','IGHM', 'IGHA1', 'IGHA2', 'IGHG1', 'IGHG2', 'IGHG3', 'IGHG4',#IgG
#                      'ISG20', 'IFITM1', 'IFI16', 'IFITM2','IRF8', #IFN
#                      'HSPA1A', 'HSPA1B', 'TUBB4B', 'GAPDH', 'MYC')

# B_function_genes
B_function_genes <- c(MHCII_genes, 'CTSD', 'TAP1', 'B2M', 'PSME1', 'PSME2',
                      'ISG20', 'IFITM1', 'IFI16', 'IFITM2','IRF8', #IFN
                      'HSPA1A', 'HSPA1B', 'TUBB4B', 'GAPDH', 'MYC')

In [ ]:
#add the Age_group_Cell_type idents
Crm2024_BCell_SeuratObj <- FetchData(Crm2024_BCell_SeuratObj, vars = c('Age_group', 'Bcluster_res0.2')) %>%
    unite(col = 'Age_group_Cell_type', c('Age_group', 'Bcluster_res0.2')) %>%
    AddMetaData(object = Crm2024_BCell_SeuratObj)

# plot
options(repr.plot.width = 12, repr.plot.height = 6)
DotPlot(Crm2024_BCell_SeuratObj, features = B_function_genes, group.by = 'Age_group_Cell_type', dot.scale = 10) +
    scale_y_discrete(limits = c('Young_Naive_B', 'Young_Memory_B', 'Old_Naive_B', 'Old_Memory_B')) +
#    scale_color_gradient(low = '#E2E1EF', high = '#C23339') +
    scale_color_gradient(low = '#E2E1EF', high = '#C23339', limits = c(-1,2), oob = scales::squish) +
    labs(x = NULL, y = NULL) +
    theme_bw(base_size = 25) +
    transparent_bg +
    theme(legend.position = 'top',
          legend.title = element_text(size = 20),
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
          axis.text.x = element_text(angle = 90),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)),
          panel.grid = element_line(colour = 'gray', linewidth = 0.5, linetype = 'dashed'))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Crm_BCell_Function_DotPlot.pdf'), device = 'pdf', width = 12, height = 6, bg = 'transparent')

In [ ]:
##find DEGs between BCell_young and BCell_old with wilcox_test
Idents(Crm2024_BCell_SeuratObj) <- Crm2024_BCell_SeuratObj$`Bcluster_res0.2`
B_subcluster <- c('Naive_B', 'Memory_B')

for ( i in B_subcluster) {
    print(i)
    assign(x = paste0('DEGs_BCell_', i, '_young_old_tumor_wilcox'), 
           value = FindMarkers(object = Crm2024_BCell_SeuratObj, ident.1 = "Young", ident.2 = 'Old', test.use = "wilcox",
                                          logfc.threshold = 0, min.pct = 0.1, group.by = 'Age_group', subset.ident = i))
    }

In [ ]:
DEGs_BCell_Naive_B_young_old_tumor_wilcox %>% 
    filter(p_val < 0.05) %>% 
    arrange(avg_log2FC) %>%
    mutate(gene = rownames(.)) %>%
    filter(gene %in% B_function_genes)

In [ ]:
DEGs_BCell_Memory_B_young_old_tumor_wilcox %>% 
    filter(p_val < 0.05) %>% 
    arrange(avg_log2FC) %>%
    mutate(gene = rownames(.)) %>%
    filter(gene %in% B_function_genes)

# APC_Cell_analysis

### Subset

In [ ]:
Idents(Crm2024_SeuratObj) <- Crm2024_SeuratObj$`Immcluster_res1`
levels(Crm2024_SeuratObj)

In [ ]:
# subset APC_Cell
Idents(Crm2024_SeuratObj) <- Crm2024_SeuratObj$`Immcluster_res1`
Crm2024_APCell_SeuratObj <- subset(Crm2024_SeuratObj, idents = c('B_Cell', 'DC', 'Macro', 'Mono'))
Crm2024_APCell_SeuratObj

### Clustring and annotation analysis

In [ ]:
# UMAP
Crm2024_APCell_SeuratObj <- RunUMAP(Crm2024_APCell_SeuratObj, dims = 1:50, reduction = "harmony", verbose = F )


In [ ]:
# FindNeighbors
Crm2024_APCell_SeuratObj <- FindNeighbors(Crm2024_APCell_SeuratObj, dims = 1:50, reduction = "harmony", verbose = F)

In [ ]:
# FindClusters
options(repr.plot.width = 8, repr.plot.height = 8)
for (i in c(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1)) {
  Crm2024_APCell_SeuratObj <- FindClusters(Crm2024_APCell_SeuratObj, resolution = i, verbose = F)
  print(DimPlot(Crm2024_APCell_SeuratObj, reduction = "umap", label = T) + labs(title = paste0("resolution: ", i)))
}

In [ ]:
options(repr.plot.height =24, repr.plot.width = 24)

imm_markers <- c('CD79A', 'MS4A1',
                 'CD38', 'SUGCT',
                 'AICDA', 'BCL6', 'JCHAIN',
                 'CD1C', 'CLEC9A', 'FSCN1', 'LILRA4',
                 'C1QC', 'CD68',
                 'FCN1', 'CD14', 'FCGR3A', 'S100A8',
                 'CSF3R', 'FCGR3B',
                 'KIT')

FeaturePlot(object = Crm2024_APCell_SeuratObj, features = imm_markers, order = T)

In [ ]:
options(repr.plot.height =8, repr.plot.width = 8)
DimPlot(Crm2024_APCell_SeuratObj, group.by = 'Immcluster_res1', label = T)

In [ ]:
#find APCMarkers_Crm2024_res0.4_presto with wilcox test in presto
APCMarkers_Crm2024_res0.4_presto <- presto::wilcoxauc(X = Crm2024_APCell_SeuratObj, group_by = 'RNA_snn_res.0.4')

In [ ]:
APCMarkers_Crm2024_res0.3_presto %>% filter(group == '7') %>% arrange(-auc) %>% head(60)

In [ ]:
#APCcluster annotation_res0.4
options(repr.plot.height = 8, repr.plot.width = 10)
Idents(Crm2024_APCell_SeuratObj) <- Crm2024_APCell_SeuratObj$`RNA_snn_res.0.4`
## rename clusters
APCcluster_anno <- c("0" = "Memory_B", "1" = "cDC2", "2" = "Naive_B", "3" = "Macro", "4" = "CD16_Mono",
                     "5" = "CD14_Mono", "6" = "LAMP3_DC", "7" = "pDC", "8" = "cDC1", "9" = "GC_B")

Crm2024_APCell_SeuratObj <- RenameIdents(Crm2024_APCell_SeuratObj, APCcluster_anno)
Crm2024_APCell_SeuratObj$APCcluster_res0.4 <- Idents(Crm2024_APCell_SeuratObj)

DimPlot(Crm2024_APCell_SeuratObj, label = T, pt.size = 0.5, label.size = 6, repel = T, reduction = 'umap', group.by = 'APCcluster_res0.4') +
    theme(plot.title = element_text(size = 30),
          legend.text = element_text(size = 20),
          legend.key.size = unit(0.5, "inches")) +
    guides(colour = guide_legend(override.aes = list(size = 5)))

## DEGs analysis

In [ ]:
##find DEGs between APC_young and APC_old with wilcox_test
Idents(Crm2024_APCell_SeuratObj) <- Crm2024_APCell_SeuratObj$`Immcluster_res1`
APC_subcluster <- c('Macro', 'Mono', 'DC', 'B_Cell')

for ( i in APC_subcluster) {
    print(i)
    assign(x = paste0('DEGs_APCell_', i, '_young_old_tumor_wilcox'), 
           value = FindMarkers(object = Crm2024_APCell_SeuratObj, ident.1 = "Young", ident.2 = 'Old', test.use = "wilcox",
                                          logfc.threshold = 0, min.pct = 0.1, group.by = 'Age_group', subset.ident = i))
    }

In [ ]:
DEGs_APCell_B_Cell_young_old_tumor_wilcox %>% 
    filter(p_val < 0.05) %>% 
    arrange(avg_log2FC) %>%
    mutate(gene = rownames(.)) %>%
    filter(gene %in% c("HLA-DRB5", "PSME2"))

In [ ]:
DEGs_APCell_DC_young_old_tumor_wilcox %>% 
    filter(p_val < 0.05) %>% 
    arrange(avg_log2FC) %>%
    mutate(gene = rownames(.)) %>%
    filter(gene %in% c("HLA-DRB5", "PSME2"))

In [ ]:
DEGs_APCell_Macro_young_old_tumor_wilcox %>% 
    filter(p_val < 0.05) %>% 
    arrange(avg_log2FC) %>%
    mutate(gene = rownames(.)) %>%
    filter(gene %in% c("HLA-DRB5", "PSME2"))

In [ ]:
DEGs_APCell_Mono_young_old_tumor_wilcox %>% 
    filter(p_val < 0.05) %>% 
    arrange(avg_log2FC) %>%
    mutate(gene = rownames(.)) %>%
    filter(gene %in% c("HLA-DRB5", "PSME2"))

In [ ]:
# Featureplot('HLA-DRB5', 'PSME2')

options(repr.plot.width = 15, repr.plot.height = 10)
FeaturePlot(Crm2024_APCell_SeuratObj, features = c('HLA-DRB5', 'PSME2'), split.by = 'Age_group', pt.size = 0.5, order = T) &
    scale_color_gradient(low = '#E2E1EF', high = '#C23339', limits = c(0,4), oob = scales::squish) &
    theme(axis.text = element_blank(),
          axis.line = element_blank(),
          axis.ticks = element_blank(),
          axis.title = element_blank(),
          legend.position = 'right')

## Signature score analysis

In [ ]:
## prepare the predefined gene set
msigdb_c2_kegg <- msigdbr(species = 'Homo sapiens', category = "C2",subcategory = "KEGG")
msigdb_c2_kegg_list <- msigdb_c2_kegg %>% split(x = .$gene_symbol, f = .$gs_description)

In [ ]:
unique(msigdb_c2_kegg_list$'Antigen processing and presentation')

In [ ]:
# Antigen_process_and_presentation
Antigen_process <- c('PSME1', 'PSME2', 'PSME3', 'TAP1', 'TAP2', 'TAPBP',
                     'B2M', 'CALR', 'CANX', 'CIITA', 'CREB1', 'CTSB', 'CTSL', 'CTSS',
                     'LGMN', 'NFYA', 'NFYB', 'NFYC', 'PDIA3', 'RFX5', 'RFXANK', 'RFXAP')

Curated_Antigen_Process <- c('PSME1', 'PSME2', 'PSME3', 'TAP1', 'TAP2', 'TAPBP', 'ERAP1', 'ERAP2',
                     'B2M', 'CALR', 'CANX', 'CTSD', 'CTSB', 'CTSL', 'CTSS',
                     'LGMN', 'PDIA3')

Antigen_presentation <- c('HLA-A', 'HLA-B', 'HLA-C', 'HLA-E', 'HLA-F', 'HLA-G',
                          'HLA-DRA', 'HLA-DRB5', 'HLA-DRB1', 'HLA-DQA1', 'HLA-DQB1',
                          'HLA-DQA2', 'HLA-DQB2', 'HLA-DOB', 'HLA-DMB', 'HLA-DMA',
                          'HLA-DOA', 'HLA-DPA1', 'HLA-DPB1', 'HLA-DPB2', 'HLA-DRB6')


In [ ]:
#Antigen_presentation_Score; Antigen_processing_Score; respective
Crm2024_APCell_SeuratObj <- AddModuleScore(object = Crm2024_APCell_SeuratObj,
                                         features = list(Antigen_presentation, Antigen_process, Curated_Antigen_Process, unique(msigdb_c2_kegg_list$'Antigen processing and presentation'), c(Antigen_presentation, Curated_Antigen_Process)),
                                         name = c('Antigen_presentation_Score', 'Antigen_process_Score','Curated_Antigen_Process_Score', 'Antigen_Processing_and_Presentation_Score', 'Antigen_Processing_Presentation_Score'))



In [ ]:
Crm2024_APCell_SeuratObj[[]] %>% colnames()

In [ ]:
Crm2024_APCell_SeuratObj$Immcluster_res1

In [ ]:
#Antigen_presentation_Score1
options(repr.plot.width = 15, repr.plot.height = 10)
FetchData(Crm2024_APCell_SeuratObj, vars = c('Immcluster_res1', 'Age_group', 'Antigen_presentation_Score1')) %>%
    mutate(Age_group = factor(Age_group, levels = c('Young', 'Old'))) %>%
    mutate(APCcluster_res0.4 = factor(Immcluster_res1,
                                      levels = c('Macro', 'Mono', 'DC', 'B_Cell'))) %>%
    ggplot(mapping = aes(x = Immcluster_res1, y = Antigen_presentation_Score1, fill = Age_group)) +
    geom_violin(linewidth=1, position = position_dodge(0.85), alpha = 0.9) +
    geom_boxplot(notch=T, width=0.2, linewidth=1, position = position_dodge(0.85)) +
    stat_compare_means(mapping = aes(group = Age_group), label = 'p.signif', method = 'wilcox.test', size = 8) +
    scale_fill_manual(values = age_group_color, guide = guide_legend(title = element_blank(), nrow = 1)) +
    geom_hline(yintercept = 0, linetype='dashed') +
    xlab(label = NULL) +
    ylab(label = 'Antigen Presentation Score') +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(legend.position = 'none',
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
          axis.text.x = element_text(angle = 0),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))

In [ ]:
#Antigen_process_Score2
options(repr.plot.width = 15, repr.plot.height = 10)
FetchData(Crm2024_APCell_SeuratObj, vars = c('Immcluster_res1', 'Age_group', 'Antigen_process_Score2')) %>%
    mutate(Age_group = factor(Age_group, levels = c('Young', 'Old'))) %>%
    mutate(APCcluster_res0.4 = factor(Immcluster_res1,
                                      levels = c('Macro', 'Mono', 'DC', 'B_Cell'))) %>%
    ggplot(mapping = aes(x = Immcluster_res1, y = Antigen_process_Score2, fill = Age_group)) +
    geom_violin(linewidth=1, position = position_dodge(0.85), alpha = 0.9) +
    geom_boxplot(notch=T, width=0.2, linewidth=1, position = position_dodge(0.85)) +
    stat_compare_means(mapping = aes(group = Age_group), label = 'p.signif', method = 'wilcox.test', size = 8) +
    scale_fill_manual(values = age_group_color, guide = guide_legend(title = element_blank(), nrow = 1)) +
    geom_hline(yintercept = 0, linetype='dashed') +
    xlab(label = NULL) +
    ylab(label = 'Antigen Process Score') +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(legend.position = 'none',
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
          axis.text.x = element_text(angle = 0),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))

In [ ]:
#Curated_Antigen_Process_Score3
options(repr.plot.width = 15, repr.plot.height = 10)
FetchData(Crm2024_APCell_SeuratObj, vars = c('Immcluster_res1', 'Age_group', 'Curated_Antigen_Process_Score3')) %>%
    mutate(Age_group = factor(Age_group, levels = c('Young', 'Old'))) %>%
    mutate(APCcluster_res0.4 = factor(Immcluster_res1,
                                      levels = c('Macro', 'Mono', 'DC', 'B_Cell'))) %>%
    ggplot(mapping = aes(x = Immcluster_res1, y = Curated_Antigen_Process_Score3, fill = Age_group)) +
    geom_violin(linewidth=1, position = position_dodge(0.85), alpha = 0.9) +
    geom_boxplot(notch=T, width=0.2, linewidth=1, position = position_dodge(0.85)) +
    stat_compare_means(mapping = aes(group = Age_group), label = 'p.signif', method = 'wilcox.test', size = 8) +
    scale_fill_manual(values = age_group_color, guide = guide_legend(title = element_blank(), nrow = 1)) +
    geom_hline(yintercept = 0, linetype='dashed') +
    xlab(label = NULL) +
    ylab(label = 'Antigen Process Score') +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(legend.position = 'none',
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
          axis.text.x = element_text(angle = 0),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))

In [ ]:
#Antigen_Processing_and_Presentation_Score4
options(repr.plot.width = 15, repr.plot.height = 10)
FetchData(Crm2024_APCell_SeuratObj, vars = c('Immcluster_res1', 'Age_group', 'Antigen_Processing_and_Presentation_Score4')) %>%
    mutate(Age_group = factor(Age_group, levels = c('Young', 'Old'))) %>%
    mutate(APCcluster_res0.4 = factor(Immcluster_res1,
                                      levels = c('Macro', 'Mono', 'DC', 'B_Cell'))) %>%
    ggplot(mapping = aes(x = Immcluster_res1, y = Antigen_Processing_and_Presentation_Score4, fill = Age_group)) +
    geom_violin(linewidth=1, position = position_dodge(0.85), alpha = 0.9) +
    geom_boxplot(notch=T, width=0.2, linewidth=1, position = position_dodge(0.85)) +
    stat_compare_means(mapping = aes(group = Age_group), label = 'p.signif', method = 'wilcox.test', size = 8) +
    scale_fill_manual(values = age_group_color, guide = guide_legend(title = element_blank(), nrow = 1)) +
    geom_hline(yintercept = 0, linetype='dashed') +
    xlab(label = NULL) +
    ylab(label = 'Antigen Process and Present Score') +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(legend.position = 'none',
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
          axis.text.x = element_text(angle = 0),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))